# AS-IS 데이터 파이프라인 — Profiling 및 DQ 구현 현황

## 1. 전체 파이프라인에서 현재 위치

```
Bronze
  │
  ▼
① 개별 Profiling   ✅ 완료
  │  원천 데이터 소스별 특성 파악
  ▼
② 통합 Profiling   ✅ 완료
  │  채널 단위 프로파일링 결과 통합
  ▼
③ DQ               ✅ 구현 완료 (현재 단계)
  │  원본 데이터 대상 Null / 형식 / 허용값 / 중복 / 이상값 검증 및 판정
  ▼
④ Mapping          ✅ 종옥님 완료 6.1
  │  Source → Target 컬럼 매핑
  ▼
⑤ Transformation    ⏳ 
  │  타입 변환 / 코드 표준화 / 값 변환 / 구조 변환
  ▼
⑥ Silver
  ├── Target
  └── Reject
```

---

## 2. Databricks 구현 구조

노트북 및 모듈별로 역할이 명확히 분리되어 있습니다.

| 구분 | 파일/모듈 | 역할 | 비고 |
|---|---|---|---|
| **Profiling** | `profiling_functions.py` | 공통 통계 계산 로직 (Null/Distinct/길이/날짜파싱 등) | 채널 독립적 공통 함수 |
| | `profiling_config.py` | 5개 소스별 설정 (경로, 업무키, 컬럼 역할 등) | 소스 추가 시 이 파일만 확장 |
| | `profiling_runner.py` | 개별 Profiling 실행 — 5개 소스를 순회하며 계산 및 저장 | 원본 데이터를 읽는 최초 단계 |
| | `integrated_profiling_builder.py` | 통합 Profiling 실행 — 개별 결과를 모아 통합 뷰 제공 | 분석 및 매핑 기초 자료 제공 |
| **DQ 검증** | `dq_config.py` | DQ 규칙 정의 (Null, 중복, 포맷, 코드 마스터 등) | 룰 및 임계값 관리 |
| | `dq_functions.py` | 규칙 유형별 검증 계산 로직 (NULL_CHECK, DUPLICATE_CHECK 등) | 공통 검증 연산 수행 |
| | `dq_runner.py` | 지정된 브론즈 테이블을 직접 스캔하여 DQ 규칙 실행 및 집계 | 원본 데이터 기반 행 단위 오류 판정 |
| | `dq_result_builder.py` | 검증 결과 데이터 가공 및 결과 테이블 적재 | PASS/FAIL 판정 및 대표 오류행 추출 |

**핵심 설계 포인트**: 
- 개별/통합 Profiling(①, ②) 은 데이터의 전체적인 지형도와 분포를 파악하는 인사이트 탐색용으로 활용됩니다.
- DQ 검증(③)은 요약된 수치 대신 브론즈 레이어의 원본 데이터셋을 직접 스캔 및 필터링하여, 실제 통과/실패 행(Row)을 정밀하게 판정하고 오류를 격리하도록 구현되어 있습니다.

---

## 3. ①, ② Profiling — 무엇을 하나

각 소스(채널)의 **원본 데이터 구조적 특성**을 수치화하고 채널 간 비교가 가능하도록 통합합니다. 

| 구분 | 주요 산출물 | 내용 |
|---|---|---|
| **개별 Profiling** | `01_summary` ~ `06_unusual_candidates` | 테이블 행/컬럼 수, Null/Blank 비율, 길이 통계, 코드 분포, 중복 건수, 날짜 선후관계 등 |
| **통합 Profiling** | `column_profile`, `categorical_distribution`, `summary` | 전 테이블 통계를 한 장으로 모아 채널별 품질 편차 및 코드 사용 현황 파악 |

**입력 및 출력**: 
- 입력: `bronze/<소스>/consultation/ingest_date=.../` 및 소스별 코드마스터 (최신 파티션 자동 탐색)
- 출력: `profiling/<테이블명>/` 및 `profiling/integrated/` (Delta + CSV)

---

## 4. ③ DQ (데이터 품질 검증) — 무엇을 하나

프로파일링을 통해 파악된 데이터 구조를 바탕으로, 브론즈 원본 테이블에 직접 엄격한 기준치(Pass/Fail)를 적용하여 **실제 품질 이슈 행을 확정**합니다.

| 구분 | 검증 내용 및 핵심 구현 항목 |
|---|---|
| **1. Null·중복 규칙** | • 필수 컬럼 Null 및 공백 검사 (`NULL_CHECK`)<br>• 고객·계약 업무키 중복(예: `consultation_id`, `LEAD_MGMT_NO`)과 완전 동일행 중복 분리 검증 (`DUPLICATE_CHECK`) |
| **2. 형식·코드 규칙** | • 전화번호, 이메일, 날짜 등 정규식 패턴 검사 (`PATTERN_CHECK`)<br>• 허용 코드 목록(Code Master)과 실제 코드값 비교 검증 (`CODE_EXISTS`) |
| **3. 실행 및 결과** | • 브론즈 테이블 직접 스캔 후 규칙별 오류 유형, 검사/오류 건수 및 오류율 집계<br>• 검증 결과(`dq_result` 테이블 적재)를 토대로 후속 Silver 정제 단계로 연계 |

**파이프라인 코드 구조( 실행은 /maps/notebook)**:
```python
maps/
│
├── notebook/                          # 실행용 Databricks 노트북
│   ├── 00_bronze_load.py         # 원천 데이터 -> Bronze 레이어 적재
│   ├── 01_profiling_run.py       # 개별 데이터 프로파일링 실행 노트북
│   ├── 02_integrated_profiling_run.py  # 통합 프로파일링 실행 노트북
│   └── 03_dq_run.py                    # DQ 검증 규칙 실행 노트북
│
└── src/                           # 핵심 비즈니스 로직 및 모듈 소스
    ├── config/                            
    │   ├── __init__.py                 # config 패키지 초기화 파일
    │   └── settings.py                 # 전역 변수 설정 파일
    │
    ├── ingest/                            
    │   └── ingest.py               # 원천 데이터 적재 관련 모듈/로직
    │
    ├── profiling/                         
    │   ├── profiling_config.py            # 소스별 설정 파일
    │   ├── profiling_functions.py         # 공통 통계 계산 로직
    │   ├── profiling_runner.py            # 개별 프로파일링 실행 러너
    │   └── integrated_profiling_builder.py  # 통합 프로파일링 빌더
    │
    └── dq/                                
        ├── dq_config.py                   # DQ 규칙 정의
        ├── dq_functions.py                # 규칙 유형별 검증 계산 로직
        ├── dq_runner.py  # DQ 검증 실행 및 최신 파티션 자동 탐색 러너
        └── dq_result_builder.py       # 검증 결과 가공 및 테이블 적재
```

---

## 5. 다음 단계 (④~⑥) 미리 보기

- **④ Mapping**: 통합 프로파일링 결과 및 코드 검증 내용을 바탕으로 Source 컬럼 → Target 컬럼(+ 표준 코드값) 매핑 정의.
- **⑤ Transformation**: 매핑표에 따른 실제 타입 변환, 코드 표준화, 구조 변환(예: 챗봇 JSON 파싱 등) 수행.
- **⑥ Silver**: DQ 검증 및 변환 결과 중 기준을 통과한 데이터는 **Target**, 실패한 데이터는 **Reject** 레이어로 분리 적재.

---

## 6. 예상 질문

**Q. 프로파일링 결과가 DQ 검증에 직접 입력으로 쓰이나요?**
- 현재 버전의 DQ 검증(`dq_runner.py`)은 정확한 행 단위 오류 판정 및 샘플 추출을 위해 **브론즈 레이어의 원본 데이터셋을 직접 스캔**하여 룰을 수행합니다. 프로파일링 결과는 데이터의 특성을 파악하고 DQ 규칙을 설계하는 **기초 인사이트(지형도)** 역할을 담당합니다.

**Q. 새로운 채널이나 테이블이 추가되면 어떻게 하나요?**
- `profiling_config.py`와 `dq_config.py`에 각각 설정 및 룰 블록을 추가해주기만 하면, 기존 러너 코드를 수정하지 않고 확장할 수 있습니다.